In [1]:
print("lets Strt")

lets Strt


In [2]:
import pandas as td

df = td.read_csv('/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv' , index_col=0)
df.head(4)

,comments,category,word_count,num_stop_words
0,family mormon never tried explain still stare ...,positive,39,13
1,buddhism much lot compatible christianity espe...,positive,196,59
2,seriously say thing first get complex explain ...,negative,86,40
3,learned want teach different focus goal not wr...,neutral,29,15


In [3]:
df = df.iloc[: , :2]

In [5]:
import warnings
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split

/mnt/WindowsD/Projects/commentlens/.commentlens/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
x = df["comments"].astype(str)
labels = df["category"].astype(str).str.strip().str.lower()
label_mapping = {"positive": 1, "negative": 2, "neutral": 0}
y = labels.map(label_mapping)
if y.isna().any():
    invalid = sorted(labels[y.isna()].unique())
    raise ValueError(f"Unsupported categories: {invalid}")

x_train_, x_test_, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )


In [ ]:
import warnings
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.INFO)

# ==========================================
# 1. Data Prep (tumhare pipeline ke exact format mein)
# ==========================================
# df = pd.read_csv("merged_comments.csv")
x = df["comments"].astype(str)
labels = df["category"].astype(str).str.strip().str.lower()
label_mapping = {"positive": 1, "negative": 2, "neutral": 0}
y = labels.map(label_mapping)

# Test set bilkul alag rakho — tuning sirf x_train par hogi
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

x_train_reset = x_train.reset_index(drop=True)
y_train_arr = y_train.reset_index(drop=True).to_numpy()

# ==========================================
# 2. Pre-compute 5 Stratified Folds (Zero Data Leakage + Fast Trials)
# ==========================================
# Har fold mein CountVectorizer sirf us fold ke train split par fit hoga
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
cv_folds = []

print("Pre-computing 5 vectorized CV folds...")
for tr_idx, val_idx in skf.split(x_train_reset, y_train_arr):
    vec = CountVectorizer(
        max_features=6000,
        ngram_range=(1, 2),
        dtype=np.float32,
    )
    X_tr = vec.fit_transform(x_train_reset.iloc[tr_idx])
    X_val = vec.transform(x_train_reset.iloc[val_idx])
    cv_folds.append((X_tr, y_train_arr[tr_idx], X_val, y_train_arr[val_idx]))
print("Folds ready!")


# ==========================================
# 3. Helper: Optuna params -> Valid LightGBM kwargs
# ==========================================
def build_lgbm_params(p):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "boosting_type": p["boosting_type"],
        "n_estimators": p["n_estimators"],
        "learning_rate": p["learning_rate"],
        "num_leaves": p["num_leaves"],
        "max_depth": p["max_depth"],
        "min_child_samples": p["min_child_samples"],
        "min_child_weight": p["min_child_weight"],
        "min_split_gain": p["min_split_gain"],
        "max_bin": p["max_bin"],
        "min_data_in_bin": p["min_data_in_bin"],
        "colsample_bytree": p["colsample_bytree"],
        "feature_fraction_bynode": p["feature_fraction_bynode"],
        "reg_alpha": p["reg_alpha"],
        "reg_lambda": p["reg_lambda"],
        "path_smooth": p["path_smooth"],
        "extra_trees": p["extra_trees"],
        "max_delta_step": p["max_delta_step"],
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    # Class imbalance strategy
    if p["class_weight_mode"] == "balanced":
        params["class_weight"] = "balanced"
    elif p["class_weight_mode"] == "none":
        params["class_weight"] = None
    else:
        params["class_weight"] = {0: 1.0, 1: p["w_pos"], 2: p["w_neg"]}

    # Boosting & data sampling constraints (taaki koi incompatible combo crash na kare)
    if p["boosting_type"] == "gbdt":
        params["data_sample_strategy"] = p["data_sample_strategy"]
        if p["data_sample_strategy"] == "goss":
            params["top_rate"] = p["top_rate"]
            params["other_rate"] = p["other_rate"]
            params["subsample"] = 1.0
            params["subsample_freq"] = 0
        else:  # bagging
            params["subsample_freq"] = p["subsample_freq"]
            params["subsample"] = p["subsample"] if p["subsample_freq"] > 0 else 1.0
    else:  # dart (GOSS aur early stopping support nahi karta)
        params["data_sample_strategy"] = "bagging"
        params["subsample_freq"] = p["subsample_freq"]
        params["subsample"] = p["subsample"] if p["subsample_freq"] > 0 else 1.0
        params["drop_rate"] = p["drop_rate"]
        params["max_drop"] = p["max_drop"]
        params["skip_drop"] = p["skip_drop"]
        params["uniform_drop"] = p["uniform_drop"]
        params["xgboost_dart_mode"] = p["xgboost_dart_mode"]

    return params


# ==========================================
# 4. Optuna Objective Function (All Key LGBM Params)
# ==========================================
def objective(trial):
    boosting_type = trial.suggest_categorical("boosting_type", ["gbdt", "dart"])

    p = {
        "boosting_type": boosting_type,
        # Core training
        "n_estimators": trial.suggest_int("n_estimators", 200, 1500, step=50)
        if boosting_type == "gbdt"
        else trial.suggest_int("n_estimators", 150, 700, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        # Tree structure & leaf constraints
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 4, 6, 8, 10, 12, 15, 20]),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-4, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        # Histogram / Binning
        "max_bin": trial.suggest_int("max_bin", 63, 511),
        "min_data_in_bin": trial.suggest_int("min_data_in_bin", 1, 10),
        # Feature subsampling
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "feature_fraction_bynode": trial.suggest_float("feature_fraction_bynode", 0.3, 1.0),
        # Regularization & split randomization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 10.0),
        "extra_trees": trial.suggest_categorical("extra_trees", [True, False]),
        "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 5.0),
        # Class weights (Negative class ko direct tune karne ke liye)
        "class_weight_mode": trial.suggest_categorical(
            "class_weight_mode", ["balanced", "none", "custom"]
        ),
    }

    if p["class_weight_mode"] == "custom":
        p["w_pos"] = trial.suggest_float("w_pos", 0.8, 1.5)
        p["w_neg"] = trial.suggest_float("w_neg", 1.2, 3.0)

    if boosting_type == "gbdt":
        strategy = trial.suggest_categorical("data_sample_strategy", ["bagging", "goss"])
        p["data_sample_strategy"] = strategy
        if strategy == "goss":
            p["top_rate"] = trial.suggest_float("top_rate", 0.1, 0.4)
            p["other_rate"] = trial.suggest_float("other_rate", 0.05, 0.4)
        else:
            p["subsample_freq"] = trial.suggest_int("subsample_freq", 0, 7)
            if p["subsample_freq"] > 0:
                p["subsample"] = trial.suggest_float("subsample", 0.5, 0.99)
    else:  # dart
        p["subsample_freq"] = trial.suggest_int("subsample_freq", 0, 7)
        if p["subsample_freq"] > 0:
            p["subsample"] = trial.suggest_float("subsample", 0.5, 0.99)
        p["drop_rate"] = trial.suggest_float("drop_rate", 0.05, 0.3)
        p["max_drop"] = trial.suggest_int("max_drop", 10, 100)
        p["skip_drop"] = trial.suggest_float("skip_drop", 0.2, 0.8)
        p["uniform_drop"] = trial.suggest_categorical("uniform_drop", [True, False])
        p["xgboost_dart_mode"] = trial.suggest_categorical("xgboost_dart_mode", [True, False])

    lgbm_params = build_lgbm_params(p)
    fold_scores = []
    best_iters = []

    for fold_idx, (X_tr, y_tr, X_val, y_val) in enumerate(cv_folds):
        model = lgb.LGBMClassifier(**lgbm_params)

        if boosting_type == "gbdt":
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
            )
            best_iters.append(model.best_iteration_)
        else:
            model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        score = f1_score(y_val, y_pred, average="macro")
        fold_scores.append(score)

        # Intermediate reporting + pruning (kharab trials jaldi stop honge)
        trial.report(float(np.mean(fold_scores)), step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    if best_iters:
        trial.set_user_attr("avg_best_iteration", int(np.mean(best_iters)))

    return float(np.mean(fold_scores))


# ==========================================
# 5. Run Optuna Study (SQLite backed -> Resume anytime!)
# ==========================================
study = optuna.create_study(
    study_name="commentlens_lgbm_full_tune",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=1),
    storage="sqlite:///commentlens_lgbm_optuna.db",
    load_if_exists=True,
)

# Trial 0 = tumhara locked baseline (taaki Optuna yahin se beat karna shuru kare)
if len(study.trials) == 0:
    study.enqueue_trial(
        {
            "boosting_type": "gbdt",
            "data_sample_strategy": "bagging",
            "subsample_freq": 0,
            "n_estimators": 400,
            "learning_rate": 0.08,
            "num_leaves": 31,
            "max_depth": -1,
            "min_child_samples": 20,
            "min_child_weight": 1e-3,
            "min_split_gain": 0.0,
            "max_bin": 255,
            "min_data_in_bin": 3,
            "colsample_bytree": 0.8,
            "feature_fraction_bynode": 1.0,
            "reg_alpha": 1e-8,
            "reg_lambda": 1e-8,
            "path_smooth": 0.0,
            "extra_trees": False,
            "max_delta_step": 0.0,
            "class_weight_mode": "balanced",
        }
    )

# Jitne trials chaho chalao (200-300 thorough tuning ke liye solid hai)
study.optimize(objective, n_trials=200, show_progress_bar=True)

print("\nBest CV Macro F1:", study.best_value)
print("Best Params:", study.best_params)

# ==========================================
# 6. Final Refit on Full x_train & Evaluation on Untouched x_test
# ==========================================
final_vectorizer = CountVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    dtype=np.float32,
)
X_train_full = final_vectorizer.fit_transform(x_train)
X_test_full = final_vectorizer.transform(x_test)

best_lgbm_params = build_lgbm_params(study.best_params)

# Agar GBDT mein early stopping trigger hui thi toh exact optimal tree count use karo
if "avg_best_iteration" in study.best_trial.user_attrs:
    best_lgbm_params["n_estimators"] = study.best_trial.user_attrs["avg_best_iteration"]

final_model = lgb.LGBMClassifier(**best_lgbm_params)
final_model.fit(X_train_full, y_train)

y_test_pred = final_model.predict(X_test_full)
print("\n=== Final Test Set Report ===")
print(classification_report(y_test, y_test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

# Production ke liye save kar lo
joblib.dump(final_vectorizer, "commentlens_cv_6000.joblib")
joblib.dump(final_model, "commentlens_lgbm_tuned.joblib")

Pre-computing 5 vectorized CV folds...
Folds ready!


[I 2026-11-25 11:28:03,712] A new study created in RDB with name: commentlens_lgbm_full_tune
Best trial: 0. Best value: 0.853083:   0%|          | 1/200 [04:58<16:29:07, 298.23s/it]

[I 2026-11-25 11:33:01,957] Trial 0 finished with value: 0.8530827573377877 and parameters: {'boosting_type': 'gbdt', 'n_estimators': 400, 'learning_rate': 0.08, 'num_leaves': 31, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'max_bin': 255, 'min_data_in_bin': 3, 'colsample_bytree': 0.8, 'feature_fraction_bynode': 1.0, 'reg_alpha': 1e-08, 'reg_lambda': 1e-08, 'path_smooth': 0.0, 'extra_trees': False, 'max_delta_step': 0.0, 'class_weight_mode': 'balanced', 'data_sample_strategy': 'bagging', 'subsample_freq': 0}. Best is trial 0 with value: 0.8530827573377877.


Best trial: 0. Best value: 0.853083:   1%|          | 2/200 [14:16<24:48:58, 451.20s/it]

[I 2026-11-25 11:42:20,257] Trial 1 finished with value: 0.8142612039745394 and parameters: {'boosting_type': 'dart', 'n_estimators': 550, 'learning_rate': 0.060099747183803134, 'num_leaves': 52, 'max_depth': 15, 'min_child_samples': 25, 'min_child_weight': 0.0008111941985431928, 'min_split_gain': 0.18340450985343382, 'max_bin': 199, 'min_data_in_bin': 6, 'colsample_bytree': 0.602361513049481, 'feature_fraction_bynode': 0.5038603981386294, 'reg_alpha': 0.00490752054869073, 'reg_lambda': 1.9835199921231646e-07, 'path_smooth': 2.9214464853521815, 'extra_trees': False, 'max_delta_step': 3.925879806965068, 'class_weight_mode': 'custom', 'w_pos': 0.8325152889039984, 'w_neg': 2.293580733422589, 'subsample_freq': 1, 'subsample': 0.531875280562787, 'drop_rate': 0.2872213843133333, 'max_drop': 97, 'skip_drop': 0.6850384088698768, 'uniform_drop': True, 'xgboost_dart_mode': True}. Best is trial 0 with value: 0.8530827573377877.


Best trial: 0. Best value: 0.853083:   2%|▏         | 3/200 [14:47<14:11:14, 259.26s/it]

... (57 outputs left out)

Best trial: 31. Best value: 0.869969:  16%|█▌        | 32/200 [5:04:31<51:24:51, 1101.74s/it]

[I 2026-11-25 16:32:35,606] Trial 31 finished with value: 0.8699687920853897 and parameters: {'boosting_type': 'gbdt', 'n_estimators': 850, 'learning_rate': 0.029474534017381555, 'num_leaves': 84, 'max_depth': -1, 'min_child_samples': 5, 'min_child_weight': 0.0017246141946940252, 'min_split_gain': 0.22215617196448484, 'max_bin': 92, 'min_data_in_bin': 7, 'colsample_bytree': 0.7977372609914426, 'feature_fraction_bynode': 0.7289183565167426, 'reg_alpha': 3.3058152888404524e-08, 'reg_lambda': 1.595543373777161e-07, 'path_smooth': 1.5454679268128713, 'extra_trees': False, 'max_delta_step': 1.5128249988720142, 'class_weight_mode': 'balanced', 'data_sample_strategy': 'bagging', 'subsample_freq': 4, 'subsample': 0.9824321670056612}. Best is trial 31 with value: 0.8699687920853897.


Best trial: 31. Best value: 0.869969:  16%|█▋        | 33/200 [5:13:46<43:29:41, 937.61s/it] 

[I 2026-11-25 16:41:50,283] Trial 32 finished with value: 0.851631822332967 and parameters: {'boosting_type': 'gbdt', 'n_estimators': 800, 'learning_rate': 0.02587444672624005, 'num_leaves': 39, 'max_depth': -1, 'min_child_samples': 21, 'min_child_weight': 0.0017240774534478913, 'min_split_gain': 0.1914363446914198, 'max_bin': 116, 'min_data_in_bin': 10, 'colsample_bytree': 0.9968907249285577, 'feature_fraction_bynode': 0.8881082812407408, 'reg_alpha': 1.1403132331562052e-07, 'reg_lambda': 2.2871120516988672e-07, 'path_smooth': 1.2421315631558165, 'extra_trees': False, 'max_delta_step': 0.820382098470732, 'class_weight_mode': 'balanced', 'data_sample_strategy': 'bagging', 'subsample_freq': 4, 'subsample': 0.9874729487101306}. Best is trial 31 with value: 0.8699687920853897.
